In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV


In [3]:
treino = pd.read_csv(r"data/train.csv")
teste = pd.read_csv(r"data/test.csv")

print("Colunas")
print("--------------------------------------")
print(treino.dtypes)
print("--------------------------------------")

treino["Transported"] = treino["Transported"].astype(int)

Colunas
--------------------------------------
PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object
--------------------------------------


In [4]:
print("Verificando valores nulos")
print("-----------------------------------------")
print(treino.isna().sum())
print("-----------------------------------------")

treino[treino['Age'] < 1]

Verificando valores nulos
-----------------------------------------
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64
-----------------------------------------


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
19,0017_01,Earth,False,G/0/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Lyde Brighttt,1
61,0067_01,Earth,True,G/10/S,PSO J318.5-22,0.0,False,0.0,0.0,0.0,0.0,0.0,Ninaha Leeves,1
86,0092_02,Earth,True,G/9/P,TRAPPIST-1e,0.0,False,0.0,0.0,NaN,0.0,0.0,Stald Hewson,1
102,0108_03,Earth,False,G/19/S,TRAPPIST-1e,0.0,NaN,0.0,0.0,0.0,0.0,0.0,Oline Handertiz,1
157,0179_02,Earth,False,G/26/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Raque Webstephrey,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8494,9074_01,Earth,True,G/1460/S,TRAPPIST-1e,0.0,NaN,0.0,0.0,NaN,0.0,0.0,Adamie Trerady,1
8584,9163_01,Earth,True,G/1477/S,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Idace Edwartizman,1
8650,9227_04,Earth,True,G/1498/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Robyny Hinglendez,1
8654,9231_02,Mars,False,F/1888/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Walls Paie,1


In [5]:
dormindo = treino[treino["CryoSleep"] == 1]
dormindo.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,1
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,1
10,0008_02,Europa,True,B/1/P,TRAPPIST-1e,34.0,False,0.0,0.0,NaN,0.0,0.0,Altardr Flatic,1
18,0016_01,Mars,True,F/5/P,TRAPPIST-1e,45.0,False,0.0,0.0,0.0,0.0,0.0,Alus Upead,1
21,0020_01,Earth,True,E/0/S,TRAPPIST-1e,1.0,False,0.0,0.0,0.0,0.0,0.0,Almary Brantuarez,0


In [6]:
treino_formatado = treino.copy()
teste_formatado = teste.copy()

despesas = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
gastos_totais_treino = treino_formatado[despesas].sum(axis=1)
gastos_totais_teste = teste_formatado[despesas].sum(axis=1)

treino_formatado.loc[(treino_formatado['CryoSleep'].isna()) & (gastos_totais_treino > 0), 'CryoSleep'] = False
treino_formatado.loc[(treino_formatado['CryoSleep'].isna()) & (gastos_totais_treino == 0), 'CryoSleep'] = True

teste_formatado.loc[(teste_formatado['CryoSleep'].isna()) & (gastos_totais_teste > 0), 'CryoSleep'] = False
teste_formatado.loc[(teste_formatado['CryoSleep'].isna()) & (gastos_totais_teste == 0), 'CryoSleep'] = True

treino_formatado["CryoSleep"].unique()

array([False, True], dtype=object)

In [7]:
terra = treino_formatado[treino_formatado["HomePlanet"] == "Earth"]
terra.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,1
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,1
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,1
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,1
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,1


In [8]:
treino_formatado['Grupo'] = treino_formatado['PassengerId'].str.split('_').str[0]
treino_formatado['Deck'] = treino_formatado['Cabin'].str.split('/').str[0]

# Criamos um "dicionário" mapeando cada Grupo para o seu Planeta conhecido
grupo_planeta = treino_formatado.groupby('Grupo')['HomePlanet'].first()

# Preenchemos os nulos do HomePlanet mapeando o Grupo do passageiro no dicionário
nulos_home = treino_formatado['HomePlanet'].isna()
treino_formatado.loc[nulos_home, 'HomePlanet'] = treino_formatado.loc[nulos_home, 'Grupo'].map(grupo_planeta)

# Para os que AINDA estão nulos após a estratégia 1
nulos_home_restantes = treino_formatado['HomePlanet'].isna()

europa_decks = ['A', 'B', 'C', 'T']
treino_formatado.loc[nulos_home_restantes & treino_formatado['Deck'].isin(europa_decks), 'HomePlanet'] = 'Europa'

treino_formatado.loc[nulos_home_restantes & (treino_formatado['Deck'] == 'G'), 'HomePlanet'] = 'Earth'

# Os pouquíssimos que sobrarem podem ser preenchidos com 'Earth' (que é a grande maioria)
treino_formatado['HomePlanet'] = treino_formatado['HomePlanet'].fillna('Earth')

In [9]:
teste_formatado['Grupo'] = teste_formatado['PassengerId'].str.split('_').str[0]
teste_formatado['Deck'] = teste_formatado['Cabin'].str.split('/').str[0]

# Criamos um "dicionário" mapeando cada Grupo para o seu Planeta conhecido
grupo_planeta_teste= teste_formatado.groupby('Grupo')['HomePlanet'].first()

# Preenchemos os nulos do HomePlanet mapeando o Grupo do passageiro no dicionário
nulos_home = teste_formatado['HomePlanet'].isna()
teste_formatado.loc[nulos_home, 'HomePlanet'] = teste_formatado.loc[nulos_home, 'Grupo'].map(grupo_planeta_teste)

# Para os que AINDA estão nulos após a estratégia 1
nulos_home_restantes = teste_formatado['HomePlanet'].isna()

europa_decks = ['A', 'B', 'C', 'T']
teste_formatado.loc[nulos_home_restantes & teste_formatado['Deck'].isin(europa_decks), 'HomePlanet'] = 'Europa'

teste_formatado.loc[nulos_home_restantes & (teste_formatado['Deck'] == 'G'), 'HomePlanet'] = 'Earth'

# Os pouquíssimos que sobrarem podem ser preenchidos com 'Earth' (que é a grande maioria)
teste_formatado['HomePlanet'] = teste_formatado['HomePlanet'].fillna('Earth')

In [10]:
# 1. Pegamos o primeiro destino válido de cada grupo
grupo_destino = treino_formatado.groupby('Grupo')['Destination'].first()

# 2. Descobrimos quem está com o Destination nulo
nulos_dest = treino_formatado['Destination'].isna()

# 3. Preenchemos os nulos "puxando" o destino do grupo mapeado
treino_formatado.loc[nulos_dest, 'Destination'] = treino_formatado.loc[nulos_dest, 'Grupo'].map(grupo_destino)


# Para os passageiros solitários que sobraram nulos, preenchemos com o destino mais comum
treino_formatado['Destination'] = treino_formatado['Destination'].fillna('TRAPPIST-1e')

In [11]:
# 1. Pegamos o primeiro destino válido de cada grupo
grupo_destino_teste = teste_formatado.groupby('Grupo')['Destination'].first()

# 2. Descobrimos quem está com o Destination nulo
nulos_dest_teste = teste_formatado['Destination'].isna()

# 3. Preenchemos os nulos "puxando" o destino do grupo mapeado
teste_formatado.loc[nulos_dest_teste, 'Destination'] = teste_formatado.loc[nulos_dest_teste, 'Grupo'].map(grupo_destino_teste)


# Para os passageiros solitários que sobraram nulos, preenchemos com o destino mais comum
teste_formatado['Destination'] = teste_formatado['Destination'].fillna('TRAPPIST-1e')

In [12]:
# 1. Calcular a mediana da coluna Age
mediana_idade = treino_formatado['Age'].median()

# Apenas para você ver qual valor o Pandas encontrou
print(f"A mediana de idade calculada foi: {mediana_idade}")

# 2. Preencher os valores nulos com essa mediana
treino_formatado['Age'] = treino_formatado['Age'].fillna(mediana_idade)

A mediana de idade calculada foi: 27.0


In [13]:
# 1. Calcular a mediana da coluna Age
mediana_idade_teste = treino_formatado['Age'].median()

# Apenas para você ver qual valor o Pandas encontrou
print(f"A mediana de idade calculada foi: {mediana_idade_teste}")

# 2. Preencher os valores nulos com essa mediana
teste_formatado['Age'] = teste_formatado['Age'].fillna(mediana_idade_teste)

A mediana de idade calculada foi: 27.0


In [14]:
colunas_gastos = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# ETAPA 1: Aplicar regras lógicas
# Criamos uma função para garantir que aplicaremos as mesmas regras no treino e no teste
def aplicar_logica_gastos(df):
    for col in colunas_gastos:
        # Regra do Sono Criogênico: Se CryoSleep é True e o gasto é nulo, preenche com 0
        df.loc[(df['CryoSleep'] == True) & (df[col].isna()), col] = 0
        
        # Regra da Idade: Se é menor de 13 anos e o gasto é nulo, preenche com 0
        df.loc[(df['Age'] < 13) & (df[col].isna()), col] = 0
    return df

treino_formatado = aplicar_logica_gastos(treino_formatado)
teste_formatado = aplicar_logica_gastos(teste_formatado)


# ETAPA 2: A Regra do Treino
for col in colunas_gastos:
    # 1. Aprende a mediana de cada gasto EXCLUSIVAMENTE nos dados de treino
    mediana_treino = treino_formatado[col].median()
    
    # 2. Aplica essa mediana para preencher os nulos que sobraram em ambos os datasets
    treino_formatado[col] = treino_formatado[col].fillna(mediana_treino)
    teste_formatado[col] = teste_formatado[col].fillna(mediana_treino)

In [15]:
# Cria uma tabela cruzando Planeta Natal x Status VIP
tabela_vip_planeta = pd.crosstab(treino_formatado['HomePlanet'], treino_formatado['VIP'], dropna=False)

print(tabela_vip_planeta)

VIP         False  True  NaN
HomePlanet                  
Earth        4617     3  118
Europa       1994   132   42
Mars         1680    64   43


In [16]:
# 1. Encontra a moda da coluna VIP no Treino 
moda_vip = treino_formatado['VIP'].mode()[0]

# 2. Preenche os nulos em ambos os datasets
treino_formatado['VIP'] = treino_formatado['VIP'].fillna(moda_vip)
teste_formatado['VIP'] = teste_formatado['VIP'].fillna(moda_vip)

treino_formatado['VIP'] = treino_formatado['VIP'].astype(bool)
teste_formatado['VIP'] = teste_formatado['VIP'].astype(bool)

In [17]:
treino_formatado = pd.get_dummies(
    treino_formatado, 
    columns=['HomePlanet', 'Destination'], 
    dtype=int
)
teste_formatado = pd.get_dummies(
    teste_formatado, 
    columns=['HomePlanet', 'Destination'], 
    dtype=int
)
# Visualizando o resultado para confirmar a transformação
treino_formatado.head()

,PassengerId,CryoSleep,Cabin,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Grupo,Deck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0001_01,False,B/0/P,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,0,0001,B,0,1,0,0,0,1
1,0002_01,False,F/0/S,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,1,0002,F,1,0,0,0,0,1
2,0003_01,False,A/0/S,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,0,0003,A,0,1,0,0,0,1
3,0003_02,False,A/0/S,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,0,0003,A,0,1,0,0,0,1
4,0004_01,False,F/1/S,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,1,0004,F,1,0,0,0,0,1


In [ ]:
treino_formatado['CryoSleep'] = treino_formatado['CryoSleep'].astype(bool)
teste_formatado['CryoSleep'] = teste_formatado['CryoSleep'].astype(bool)

# 1. Definimos a lista de colunas de texto/identificadores para remover
colunas_inuteis = ['PassengerId', 'Name', 'Cabin', 'Grupo', 'Deck']

y_treino = treino_formatado['Transported'].astype(int)

# 2. Criamos o X (Features) removendo as colunas inúteis E a coluna alvo
X_treino = treino_formatado.drop(columns=colunas_inuteis + ['Transported'])
X_teste = teste_formatado.drop(columns=colunas_inuteis)



X_treino.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,False,39.0,False,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,1
1,False,24.0,False,109.0,9.0,25.0,549.0,44.0,1,0,0,0,0,1
2,False,58.0,True,43.0,3576.0,0.0,6715.0,49.0,0,1,0,0,0,1
3,False,33.0,False,0.0,1283.0,371.0,3329.0,193.0,0,1,0,0,0,1
4,False,16.0,False,303.0,70.0,151.0,565.0,2.0,1,0,0,0,0,1


In [ ]:
# 1. Definir o modelo base 
xgb_modelo = xgb.XGBClassifier(random_state=42)

# 2. Criar a "grade" de testes (dicionário de hiperparâmetros)
param_grid = {
    'n_estimators': [80, 100, 120],          
    'learning_rate': [0.01, 0.05, 0.1, 0.2], 
    'max_depth': [3, 5, 7],                  # Profundidade
    'subsample': [0.8, 1.0],                 # 80% ou 100% das linhas
    'colsample_bytree': [0.8, 1.0]           # 80% ou 100% das colunas
}

# 3. Configurar o validador cruzado 
grid_search = GridSearchCV(
    estimator=xgb_modelo, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1,          
    verbose=1           
)

# 4. Iniciar o treinamento 
print("Iniciando a busca pelos melhores parâmetros...")
grid_search.fit(X_treino, y_treino)

# 5. Revelar os resultados
print("\nMelhores parâmetros encontrados:")
print(grid_search.best_params_)
print(f"Melhor acurácia no Cross-Validation: {grid_search.best_score_:.4f}")

Iniciando a busca pelos melhores parâmetros...
Fitting 5 folds for each of 144 candidates, totalling 720 fits

Melhores parâmetros encontrados:
{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
Melhor acurácia no Cross-Validation: 0.8016


In [20]:
# 1. Resgatamos o modelo campeão do GridSearch
melhor_modelo = grid_search.best_estimator_

# 2. Fazemos as previsões com os dados de teste 
previsoes = melhor_modelo.predict(X_teste)

# 3. O Kaggle exige que a coluna seja True/False
previsoes_bool = previsoes.astype(bool)

# 4. Criamos o DataFrame final juntando os IDs
submissao = pd.DataFrame({
    'PassengerId': teste['PassengerId'], 
    'Transported': previsoes_bool
})

# 5. Exportamos para CSV 
submissao.to_csv('submissao_xgboost.csv', index=False)

print("Arquivo gerado com sucesso! Verifique a pasta do seu projeto.")

Arquivo gerado com sucesso! Verifique a pasta do seu projeto.
